In [1]:
from datetime import date, datetime
from pathlib import Path

import pandas as pd

from rec_gov_automate import FourRivers, get_fourrivers_availability
from rec_gov_automate.utils.notification import send_pushover

In [2]:
search_data = Path.cwd().parent / 'data' / 'external' / 'four_rivers_search.csv'
search_data = Path.cwd().parent / 'testing' / 'four_rivers_search_test.csv'

assert search_data.exists()

## Load Search Rivers and Dates

Load the rivers and dates to search Recreation.gov for from a comma separated values file with a specific schema.

In [3]:
search_df = pd.read_csv(search_data)

search_df

,search_group,river_key,month,day,putin_id,takeout_id,permit_pickup_id,days,checkout,recgov_user,recgov_pass
0,0,selway,7,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,middle_fork,7,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0,hells_canyon,7,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0,main_salmon,7,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0,main_salmon,12,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,0,main_salmon,12,26,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Retrieve Available Permits

Search Recreation.gov for dates with avaiable permits and return only available dates.

In [4]:
avail_df = get_fourrivers_availability(search_df=search_df)

avail_df

,search_group,river_key,putin_id,takeout_id,permit_pickup_id,days,checkout,recgov_user,recgov_pass,date,total,remaining
0,0,main_salmon,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-12-25 00:00:00+00:00,99,99
1,0,main_salmon,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-12-26 00:00:00+00:00,99,99


## Notify of Availabilty

Create notificaiton string and provide notification of available dates.

In [5]:
notify_str = "Four Rivers permits are available!\n\n"

for _, river_key, launch_dt, permit_cnt in avail_df[['river_key', 'date', 'remaining']].itertuples():

    # format the name of the river
    river_name = river_key.replace('_', ' ').title()

    # format the date
    dt_str = launch_dt.strftime('%a %d%b')

    # assemble the string
    avail_str = f'{river_name}: {dt_str} - {permit_cnt} permits\n'

    # add the string
    notify_str = notify_str + avail_str

print(notify_str)

send_pushover(notify_str)

Four Rivers permits are available!

Main Salmon: Thu 25Dec - 99 permits
Main Salmon: Fri 26Dec - 99 permits



<Response [200]>